## Demos of parsing various formats
This demonstrates import and parsing of various formats. All of these examples are from a single file, but the import can take a list of files to concatenate data from multiple files
</br></br>
J.D. Landgrebe, Data Delve LLC</br>
September 18, 2026

In [1]:
# Import standard libraries and instance helper classes
import pandas as pd, numpy as np
from libs.import_classes import instance_classes
from libs.projtables import Table

# instance_classes minimizes user-facing code for instancing standard, toolbox-related classes
files, tbls = instance_classes(IsTest=True, subdir_tests='test_data_parse')

## Survey Data Example
Import and parse a file where its single sheet contains multiple rows/columns data blocks --one per survey question. The survey question textis adjacent to the data and is included as a key column by specifying a "BlockID" string
</br></br>
<img src="./images/survey_data.png" width="500">
</br></br>
The parsing approach is to search for occurrences of the string, "Answer Choices" that precedes each block and include data rows until a blank is encountered. This provides flexibility for questions having a varying number of responses. The "Question" column in parsed data is created as a BlockID variable by specifying a tuple: (new column name, row offset from start of data, column index)

#### Instance Survey Table with Import and Parsing Instructions

In [2]:
# Set dictionary of import parameters
dImportParams = {'ftype':'excel', 'import_path':files.path_data,'sht':'raw_table'}

# Dictionary of parse parameters for this raw data
dParseParams = {}
dParseParams['is_unstructured'] = True
dParseParams['parse_type'] = 'RowMajorTbl'
dParseParams['import_dtype'] = str
dParseParams['flag_start_bound'] = 'Answer Choices'
dParseParams['flag_end_bound'] = '<blank>'
dParseParams['icol_start_bound'] = 0
dParseParams['icol_end_bound'] = 0
dParseParams['iheader_rowoffset_from_flag'] = 0
dParseParams['idata_rowoffset_from_flag'] = 1

#Block ID variable for question text
dParseParams['block_id_vars'] = ('Question', -2, 0)

# Instance the survey Table object with the import and parse dicts
tbls.survey = Table('tbl_survey', 
                    dImportParams=dImportParams, 
                    dParseParams=dParseParams)

type(tbls.survey).__name__, type(tbls).__name__

('Table', 'ProjectTables')

#### Import and parse the raw data

In [3]:
# Import to a list (one item here) of raw data df's from specified file(s)
tbls.survey.ImportToTblDf(lst_files='tbl1_survey.xlsx')

# Parse the raw data into tbl.df structured Dataframe
tbls.survey.ParseRawData()
tbls.survey.df

,Question,Answer Choices,Response Percent,Responses,1,2,3
0,Q1. How often do you wash your car?,Daily,14.13%,76,NaN,NaN,NaN
1,Q1. How often do you wash your car?,A few times a week,41.82%,225,NaN,NaN,NaN
2,Q1. How often do you wash your car?,Weekly,36.62%,197,NaN,NaN,NaN
3,Q1. How often do you wash your car?,A few times a month,7.06%,38,NaN,NaN,NaN
4,Q1. How often do you wash your car?,Rarely,0.37%,2,NaN,NaN,NaN
5,Q2. What brands of car wash cleaner do you use,Turtle Wax,46.45%,250,NaN,NaN,NaN
6,Q2. What brands of car wash cleaner do you use,Dawn Dishwashing Detergent,27.90%,150,NaN,NaN,NaN
7,Q2. What brands of car wash cleaner do you use,Other (please specify),25.65%,138,NaN,NaN,NaN
8,Q3. How would you improve your current product...,Lower price point,NaN,NaN,91,33,19
9,Q3. How would you improve your current product...,Better package,NaN,NaN,31,37,16


## Multisheet Excel Example
Import and parse a data on multiple sheets in Excel file(s). Because the data are structured, rows/columns, no separate parsing step is needed. Instead of specifying a sheet name as in the previous example, we specify a `sht_type='all'` import paramger to instruct importing from all sheets in each workbook.  Because no `IsUnstructured` param is specified (it is False by default), import directly adds data to tbl.df.
</br></br>
<img src="./images/multisheet.png" width="500">
</br></br>

In this case, the Parse Params are simply to specify the format as `RowMajorTbl`. The `'import_dtype'=str` negates inferring float data type for integers and `NaN` for blanks (in lieu of None values for blanks) The default `IsUnstructured=False` denotes that the data are rows/columns arrangement with header row 1 and data beginning in row 2.  We add the `add_filename_col` param to cause the import to add filename and sheet columns to the data allowing precise tracking of the data source.

In [4]:
# Set dictionary of import parameters. sht_type='all"
dImportParams = {'ftype':'excel', 'import_path':files.path_data, 'sht_type':'all'}

# Dictionary of parse parameters for this raw data
dParseParams = {'parse_type': 'RowMajorTbl', 'import_dtype': str}

# Specify adding filename and sheet columns as keys in the parsed data
dParseParams['add_filename_col'] = True

# Instance the survey Table object with the import and parse dicts
tbls.multisheet = Table('tbl_multisheet', 
                    dImportParams=dImportParams, 
                    dParseParams=dParseParams)

#### Import the raw data (no parsing needed)

In [9]:
# Import raw data to tbl.df
tbls.multisheet.ImportToTblDf(lst_files='multisheet.xlsx')
tbls.multisheet.df

,filename,sheet,idx_raw,col_1,col_2,idx,extra_1,extra_2
0,multisheet.xlsx,first_sheet,1.0,10,a,NaN,NaN,NaN
1,multisheet.xlsx,first_sheet,2.0,20,b,NaN,NaN,NaN
2,multisheet.xlsx,first_sheet,3.0,30,c,NaN,NaN,NaN
3,multisheet.xlsx,first_sheet,4.0,40,d,NaN,NaN,NaN
4,multisheet.xlsx,first_sheet,5.0,50,e,NaN,NaN,NaN
5,multisheet.xlsx,second_sheet,NaN,100,aa,6.0,NaN,NaN
6,multisheet.xlsx,second_sheet,NaN,200,bb,7.0,NaN,NaN
7,multisheet.xlsx,second_sheet,NaN,300,cc,8.0,NaN,NaN
8,multisheet.xlsx,second_sheet,NaN,400,dd,9.0,NaN,NaN
9,multisheet.xlsx,second_sheet,NaN,500,ee,10.0,NaN,NaN


## Column Major Raw Data
Import and parse data where the "observations" are arranged left to right in columns with labels in a column to the left of the data and where the raw data are bounded by a Totals row at the bottom of the block. This format comes up commonly for time-based data where it is aesthetically nice to have data arranged left to right in time order.
</br></br>
<img src="./images/colmajor.png" width="550">
</br></br>

In [6]:
# dictionary of import parameters
dImportParams = {'ftype':'excel', 'import_path':files.path_data,'sht':'Sheet1'}

# Dictionary of parse parameters for this raw data
dParseParams = {}
dParseParams['is_unstructured'] = True
dParseParams['parse_type'] = 'ParseColMajorTbl'
dParseParams['import_dtype'] = str
dParseParams['flag_start_bound'] = 'Total Orders'
dParseParams['flag_end_bound'] = 'Total'
dParseParams['icol_start_flag'] = 0    # 'Total Orders' is in column 0
dParseParams['icol_end_flag'] = 0  # 'Total' is in column 0
dParseParams['nrows_header_offset_from_flag'] = 1    # header row is 1 row below start flag
dParseParams['nrows_data_offset_from_flag'] = 2     # data starts 2 rows below start flag
dParseParams['nrows_data_end_offset_from_flag'] = -1  # data ends at the row above 'Total'

# Instance the survey Table object with the import and parse dicts
tbls.colmajor = Table('colmajor', 
                    dImportParams=dImportParams, 
                    dParseParams=dParseParams)

In [7]:
tbls.colmajor.ImportToTblDf(lst_files='col_major_boutique_data.xlsx')
tbls.colmajor.ParseRawData()

# Set date format
tbls.colmajor.df['col_header'] = \
    pd.to_datetime(tbls.colmajor.df['col_header']).dt.strftime('%Y-%m-%d')

tbls.colmajor.df.head(20)

,col_header,category,value
0,2026-01-25,blouses,1304.12
1,2026-01-25,skirts,1322.05
2,2026-01-25,jewelry,487.37
3,2026-01-25,purses,251.68
4,2026-02-01,blouses,1131.4
5,2026-02-01,skirts,1137.98
6,2026-02-01,jewelry,911.77
7,2026-02-01,purses,339.91
8,2026-02-08,blouses,1321.38
9,2026-02-08,skirts,1820.85
